In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Однажды мы с вами сравнивали метрики между двумя моделями на одной и той же выборке. 

Чаще всего метрики по объектам представляют собой ненормальное распределение, поэтому лучше оценивать их различия непараметрическими статистическими критериями. Давайте попрактикумемся в этом. Возьмём уже привычный нам датасет со средними ценами на дома.

Обучите Ridge-регрессии из библиотеки sklearn с параметрами alpha=1 и alpha=10. Предсказания сделайте на всей выборке по фолдам с помощью функции cross_val_predict (по очереди каждый фолд будет становиться тестовым), не забудьте передать туда наше разбиение.

Посчитайте MSE по каждому объекту выборки и сравните равенство их медиан с помощью критерия знаков. Сам критерий знаков реализуйте вручную, его нулевое распределение представляет собой биномиальное распределение, с котором мы уже работали ранее.

Какой вывод вы можете сделать о различии этих моделей? Альтернатива двусторонняя, уровень значимости 0.05.

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing(as_frame=True, return_X_y=True)
df, target = data[0], data[1] 
df = StandardScaler().fit_transform(df)

cv = KFold(n_splits=4, shuffle=True, random_state=6)

Сделаем предсказания моделями, обученными по k-фолдовой кросс-валидации:

In [3]:
from sklearn.linear_model import Ridge, Lasso

y_ridge_1 = cross_val_predict(Ridge(alpha=1.0), X=df, y=target, cv=cv)
y_ridge_2 = cross_val_predict(Ridge(alpha=10.0), X=df, y=target, cv=cv)

Оценим качество:

In [4]:
mse_ridge_1 = (y_ridge_1 - target) ** 2
mse_ridge_2 = (y_ridge_2 - target) ** 2

np.mean(mse_ridge_1), np.mean(mse_ridge_2)

(0.5305334005667641, 0.5304527930834362)

Перейдём к новой случайной величине для критерия знаков:

In [5]:
values = (mse_ridge_1 > mse_ridge_2).astype(int)

np.mean(values)

0.4810077519379845

Оценим значимость (здесь мы учитываем, что среднее значение в выборке меньше 0.5, то есть наша статистика слева от центра):

In [6]:
from scipy.stats import binom

binom(n=len(values), p=0.5).cdf(sum(values)) * 2

5.0154276353317846e-08

Ну или автоматически:

In [ ]:
from scipy.stats import binom_test

binom_test(x=sum(values), n=len(values))

Некоторые непараметрические статистические критерии делают переход к рангам.

Посчитайте ранги для элементов следующей выборки. Ранги равных по значениям элементов считайте так же, как мы это делали на уроке.


Также можно вычислять автоматически (pandas тоже умеет). Важно выбрать в аргументе то, что делать с равными значениями.

In [8]:
import scipy.stats as ss

ss.rankdata([7, 1, 5, 1, 3, 2, 5], method='average')

array([7. , 1.5, 5.5, 1.5, 4. , 3. , 5.5])

В чём неудобство работы с табличными распределениями?

Табличное распределение необходимо посчитать для всех возможных значений и хранить большое количество значений. При этом посчитать точно всё-таки возможно и использовать его можно (не только теоретическая конструкция).

### 9

Вернёмся к уже привычному для нас датасету со средними ценами на дома, с которым мы работали в прошлых заданиях. Для скачивания воспользуйтесь тем же кодом:

В прошлом уроке мы сравнивали средние доходы (MedInc) между районами, где дома моложе 30 лет (включительно) и старше 30 лет (HouseAge). Однако тогда мы заметили, что случайная величина распределена не нормально.

Давайте воспользуемся непараметрическими статистическими критериями, которые не делают предположений на вид распределения.

Примените критерий Манна-Уитни-Уилкоксона к выборке и сравните, различаются ли распределения между выборками. Какой вывод вы можете сделать? Альтернатива двусторонняя.

In [9]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True, return_X_y=True)
df, target = data[0], data[1] 

Снова разделим выборку на 2:

In [10]:
df1 = df[df.HouseAge <= 30]
df2 = df[df.HouseAge > 30]